# U04 · PyTorch 基础

**核心认知**：PyTorch ≈ NumPy + 自动求导 + GPU

你 U3 学的 90% 都能直接迁移。这一单元只学三个新东西：
1. `torch.Tensor`（替换 `np.ndarray`）
2. `autograd`（自动求导，告别手推梯度）
3. `nn.Module` + `optimizer`（封装训练流程）


## §1 Tensor：PyTorch 的 ndarray

几乎所有 NumPy 操作 PyTorch 都有同名版本。


In [4]:
import torch
import numpy as np

# 创建（和 NumPy 完全平行）
x = torch.tensor([[1, 2], [3, 4]])      # 像 np.array
z = torch.zeros(2, 3)
o = torch.ones(2, 3)
r = torch.randn(2, 3)                   # 标准正态
u = torch.rand(2, 3)                    # 均匀 [0,1)
a = torch.arange(0, 10)                 # 像 np.arange

print('x:', x.shape, x.dtype)
print('x:\n', x)
print('z:\n', z)
print('o:\n', o)
print('r:\n', r)
print('u:\n', u)
print('a:\n', a)


x: torch.Size([2, 2]) torch.int64
x:
 tensor([[1, 2],
        [3, 4]])
z:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
o:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
r:
 tensor([[ 0.0832,  0.3149, -2.9413],
        [-0.1278, -0.2181, -0.5340]])
u:
 tensor([[0.3981, 0.6978, 0.6713],
        [0.4840, 0.1609, 0.3384]])
a:
 tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [6]:
# shape 操作（和 NumPy 一模一样）
x = torch.arange(12)
print(x.reshape(3, 4))
print(x.view(3, 4))     # view 是 PyTorch 特有，等价于 reshape（前提是内存连续）
print(x.reshape(3, 4).T)
print(x[:, None].shape) # 升维一样用
print(x.shape)


tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([[ 0,  4,  8],
        [ 1,  5,  9],
        [ 2,  6, 10],
        [ 3,  7, 11]])
torch.Size([12, 1])
torch.Size([12])


## §2 NumPy ↔ PyTorch 互转

用 `torch.from_numpy` 和 `.numpy()` 来回切。注意：默认**共享内存**！


In [7]:
import numpy as np
import torch

# NumPy → Torch
a = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(a)
print(t)

# Torch → NumPy
back = t.numpy()
print(back)

# 共享内存的坑：改 t 也会改 a
t[0] = 999
print('a 也变了:', a)

# 想独立：先 .clone()
t2 = torch.from_numpy(a).clone()


tensor([1., 2., 3.], dtype=torch.float64)
[1. 2. 3.]
a 也变了: [999.   2.   3.]


## §3 GPU：`.to(device)`

Tensor 有一个属性叫 `device`，决定它住在 CPU 还是 GPU 上。
GPU 上的运算才能用上你显卡的并行能力。

**规则**：参与同一个运算的 tensor 必须在**同一设备**。


In [9]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('当前设备:', device)

x = torch.randn(3, 4)
print('默认设备:', x.device)

x = x.to(device)        # 搬过去
print('搬到:', x.device)

# 你 CPU-only 也没关系，写 .to(device) 是好习惯，未来切 GPU 不用改代码


当前设备: cpu
默认设备: cpu
搬到: cpu


## §4 autograd：自动求导（核心！）

U2 你手推链式法则，U3 你手写 `dw = 2*(x*(y_pred-y)).mean()`。
**从今天起再也不用手推了**——PyTorch 会自动算。

三步：
1. 创建张量时设 `requires_grad=True`，告诉 PyTorch「我要追踪这个变量」
2. 用它做运算得到 loss
3. 调 `loss.backward()`，梯度自动出现在 `.grad` 里


In [10]:
import torch

# 例子：f(x) = x^2，求 df/dx 在 x=3 处
x = torch.tensor(3.0, requires_grad=True)
f = x ** 2

f.backward()         # 自动反向传播
print('df/dx =', x.grad)   # 期望 2*3 = 6


df/dx = tensor(6.)


In [11]:
# 多变量例子：z = x*y + x^2，求 dz/dx 和 dz/dy 在 (2, 3)
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

z = x * y + x ** 2
z.backward()

print('dz/dx =', x.grad)   # 期望 y + 2x = 3 + 4 = 7
print('dz/dy =', y.grad)   # 期望 x = 2


dz/dx = tensor(7.)
dz/dy = tensor(2.)


### autograd 的两个坑

**坑1：梯度会累加**。每次 `backward()` 都把梯度累加到 `.grad` 上，所以训练循环里要先 `optimizer.zero_grad()`（或手动 `x.grad.zero_()`）清零。

**坑2：参数更新要在 `torch.no_grad()` 里做**。否则更新本身也会被追踪进计算图，造成内存泄漏。或者直接用 `optimizer.step()`（推荐）。


In [12]:
# 演示坑1：梯度累加
x = torch.tensor(3.0, requires_grad=True)

(x**2).backward()
print('第1次:', x.grad)   # 6

(x**2).backward()
print('第2次:', x.grad)   # 12！累加了

x.grad.zero_()           # 清零
(x**2).backward()
print('清零后:', x.grad)  # 6


第1次: tensor(6.)
第2次: tensor(12.)
清零后: tensor(6.)


## §5 nn.Module：封装一层 / 一个网络

U3 你写 `y = x @ W + b`，自己 init `W` 和 `b`。
PyTorch 把这种「带参数的运算」封装成 `nn.Linear` 一行搞定。

**nn.Module 的本质**：一个对象，里面装了「参数 + forward 函数」。


In [13]:
import torch.nn as nn

# nn.Linear(in_features, out_features) 内部自动管理 W 和 b
layer = nn.Linear(10, 3)

x = torch.randn(32, 10)   # batch=32
y = layer(x)              # 等价于 x @ W.T + b
print('输出:', y.shape)   # (32, 3)

# 看看里面的参数
for name, p in layer.named_parameters():
    print(name, p.shape, 'requires_grad:', p.requires_grad)


输出: torch.Size([32, 3])
weight torch.Size([3, 10]) requires_grad: True
bias torch.Size([3]) requires_grad: True


In [14]:
# 自定义网络：继承 nn.Module，写 __init__ 和 forward
import torch.nn as nn
import torch.nn.functional as F

class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        self.fc2 = nn.Linear(20, 3)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

net = MyNet()
x = torch.randn(32, 10)
print('输出:', net(x).shape)
print('参数总数:', sum(p.numel() for p in net.parameters()))


输出: torch.Size([32, 3])
参数总数: 283


## §6 优化器：自动更新参数

U3 你手写 `w = w - lr * dw`。
PyTorch 把这步交给优化器：

```python
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
...
optimizer.step()    # 自动 w = w - lr * w.grad
```

常用优化器：
- `SGD`：最朴素的梯度下降
- `Adam`：自适应学习率，**99% 场景的默认选择**


## §7 完整训练循环模板（背下来！）

U3 五步：前向 → loss → 梯度 → 更新 → 清零

U4 PyTorch 三行核心：
```python
for epoch in range(epochs):
    optimizer.zero_grad()        # 1. 清零旧梯度
    y_pred = model(x)            # 2. 前向
    loss = loss_fn(y_pred, y)    # 3. 算 loss
    loss.backward()              # 4. 自动算梯度
    optimizer.step()             # 5. 自动更新
```

**这个模板从现在到 GRU、Transformer 都不会变，只会换里面的 model。**


In [19]:
# 完整 demo：用 nn.Module + 优化器 重做线性回归
import torch
import torch.nn as nn

torch.manual_seed(42)

# 数据
x = torch.linspace(-5, 5, 100).unsqueeze(1)         # (100, 1)
y = 2 * x + 3 + torch.randn(100, 1) * 0.5

# 模型
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.03)

# 训练
for epoch in range(200):
    optimizer.zero_grad()
    y_pred = model(x)
    loss = loss_fn(y_pred, y)
    loss.backward()
    optimizer.step()
    if epoch % 40 == 0:
        w = model.weight.item()
        b = model.bias.item()
        print(f'epoch {epoch:3d}  loss={loss.item():.4f}  w={w:.3f}  b={b:.3f}')

print('\n最终参数:')
print('w =', model.weight.item(), '(理论 2.0)')
print('b =', model.bias.item(),   '(理论 3.0)')


epoch   0  loss=44.3319  w=1.095  b=-0.645
epoch  40  loss=0.3490  w=1.997  b=2.721
epoch  80  loss=0.2415  w=1.997  b=3.004
epoch 120  loss=0.2407  w=1.997  b=3.028
epoch 160  loss=0.2407  w=1.997  b=3.030

最终参数:
w = 1.9970524311065674 (理论 2.0)
b = 3.0298662185668945 (理论 3.0)


## 一句话总结

| U3 (NumPy) | U4 (PyTorch) |
|---|---|
| 手推梯度 | `loss.backward()` |
| `w = w - lr * dw` | `optimizer.step()` |
| 自己写 `forward` | `nn.Linear` / 自定义 `nn.Module` |
| CPU only | `.to('cuda')` 切 GPU |

做完 `exercises.ipynb` 再继续。
